In [ ]:
import pandas as pd
import numpy as np
np.random.seed(21) #фиксируем начальное состояние генератора для воспроизводимости результатов
from nltk import word_tokenize, FreqDist, sent_tokenize
from pymorphy3 import MorphAnalyzer
from string import punctuation
from nltk.corpus import stopwords
import re

In [ ]:
df_res=pd.read_excel('ребра итогового графа с весами.xlsx', sheet_name='Sheet1')
df_users=pd.read_excel('пользователи из города.xlsx', sheet_name='Sheet1')

In [ ]:
morph = MorphAnalyzer()
stop_w = list(punctuation) + stopwords.words('russian') + ['`', '..', '"', '...', '»', '«', '``', "''", 'это', 'который', 'ещё', '|https', 'либо', 'всё', 'весь', 'свой']

In [ ]:
#функция сортировки вершин по кластерам
def SortingByClusters(clusters):
	clusters_distribution = [] #список кортежей результата
	uniq_cluster = clusters['cluster'].unique() #количество уникальных кластеров
	for cl in uniq_cluster:
		ver_in_clusters = [] #список вершинн кластера
		for IDRow in range(clusters.shape[0]):
			if clusters['cluster'][IDRow]==cl:
				ver_in_clusters.append(clusters['id'][IDRow])
		clusters_distribution.append((cl, ver_in_clusters))
	return(clusters_distribution)

In [ ]:
#функция поиска общих слов в постах пользователей из кластера
def SearchForCommonWords(UsersInClusters):
    PostWordsOfClusters = [] #список слов в текстах кластера
    common_words = [] #общие слова встречаемые слова в текстах постов кластера
    count_of_word = 0
    for IDUsers in UsersInClusters:
        #для пользователя находим тексты его постов, переводим текст в нижний регистр, разделяем на слова
        for UsersRow in range(df_users.shape[0]):
            if df_users['id'][UsersRow]==IDUsers:
                #если слово не является стоп-словом, знаком препинания или числом, переводим его в нормальную форму и записываем в список слов постов
                user_post_sent = sent_tokenize(df_users['post'][UsersRow].lower())
                post_w = set()
                for post_sent in user_post_sent:
                    post_word = word_tokenize(post_sent)
                    for word in post_word:
                       normal_w = morph.parse(word)[0].normal_form
                       if normal_w not in stop_w and not re.match(r'\d+', normal_w):
                           post_w.add(normal_w)
                #как пройдены все предложения текста постов пользователя, добавляем список слов текстов постов в список слов текстов кластера
                PostWordsOfClusters.append(post_w)
                count_of_word = count_of_word + len(post_w)
    
    #находим пересечение множеств слов из текстов                      
    code = str(PostWordsOfClusters[0]) #начальное множество, множество слов первого текста
    for SetPostWords in PostWordsOfClusters[1:]:
         code = code + " & " + str(SetPostWords) #заполняем выражение для нахождения объединения множества слов
    common_words = eval(code)
    return((common_words, count_of_word))

In [ ]:
#поиск часто встречающихся слов
def SearchForFrequentWords(UsersInClusters):
    PostWordsOfClusters = [] #список слов кластера
    common_words = [] #часто встречаемые слова в кластере
    count_of_words = 0
    for IDUsers in UsersInClusters:
        #для пользователя находим тексты его постов, переводим текст в нижний регистр, разделяем на слова
        for UsersRow in range(df_users.shape[0]):
            if df_users['id'][UsersRow]==IDUsers:
                user_post_sent = sent_tokenize(df_users['post'][UsersRow].lower())
                for post_sent in user_post_sent:
                    post_word = word_tokenize(post_sent)
                    count_of_words = count_of_words + len(post_word)
                    for s_word in post_word:
                        normal_w = morph.parse(s_word)[0].normal_form
                        #если слово не являетсся стоп-словом, знаком препинаия или числом, записяваем в список слов кластера
                        if normal_w not in stop_w and not re.match(r'\d+', s_word):
                            PostWordsOfClusters.append(normal_w)
                    
    FreqDistOfClusters = FreqDist(PostWordsOfClusters)            
    common_words = FreqDistOfClusters.most_common(1)
    return((common_words, count_of_words))

In [ ]:
#функция для генерации цвета кластеров
def ColorsGeneration(unique_label):
	colors=pd.DataFrame(columns=['mean', 'color'])
	colors['mean']=unique_label
	color=[] #лист цветов
	no_unique=False
	for col in unique_label:
		while no_unique!=True:
			RGB=list(np.random.random(size=3))
			RGB.append(1)
			if (RGB in color): no_unique=False
			else: no_unique=True
		color.append(RGB)
		no_unique=False
	colors['color']=color
	return(colors)

In [ ]:
#функция назначения цвета каждой вершине
def NodeColor(colors, algorithm_result):
	node_color=[]
	for n in algorithm_result.iloc[:, 1]:
		for c in range(colors.shape[0]):
			if n==colors.iloc[c, 0]: node_color.append(colors.iloc[c, 1])
	return(node_color)

In [ ]:
#функция визуализации графа с помощью igraph и matplotlib
def VisualizationOfIgraph(I, node_color, file_name, graph_title):
    layout = I.layout("kk")
    I.vs["color"]=node_color
    l_weight_visualization = [f"{weight_for_formatting:.2f}" for weight_for_formatting in df_res["weight"]]
    
    fig, ax = plt.subplots(figsize=(39, 45)) #объединение инициализации области для рисунка (fig) и осей (ax)
    ig.plot(I, vertex_label=I.vs["name"], vertex_size=24, vertex_label_size=14, target=ax, vertex_color=I.vs["color"], edge_label=l_weight_visualization, edge_label_size=14, layout=layout)
    
    plt.title(graph_title, fontsize=14, fontweight='bold')
    plt.savefig(file_name, bbox_inches="tight")
    plt.close()
    print("Сохранено")

In [ ]:
#функция для записи кластера каждой вершины
def ClusterForVer(partition):
	cl=0
	ver=[]
	classes=[]
	for i in partition:
		cl=cl+1
		for j in i:
			ver.append(int(j))
			classes.append(cl)
	clusters['id']=ver
	clusters['cluster']=classes
	return(clusters)

In [ ]:
#сортировка вершин и кластеров, как в списке вершин vr
def SortVer(clusters):
	ver=[]
	communities=[]
	for user_id in I.vs['name']:
		for row in range(len(clusters)):
			if (int(user_id)==clusters['id'][row]):
				ver.append(clusters['id'][row])
				communities.append(clusters['cluster'][row])
	clusters['id']=ver
	clusters['cluster']=communities
	return(clusters)

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.config("spark.jars.packages", "io.graphframes:graphframes-spark4_2.13:0.10.0").getOrCreate()
from graphframes import *
import matplotlib.pyplot as plt
%matplotlib inline
import igraph as ig

In [ ]:
all_nodes = pd.concat([df_res['src'], df_res['dst']]).unique()
g_v = list(all_nodes)
g_v=pd.DataFrame({'id': g_v})
g_v=spark.createDataFrame(g_v)
g_e=pd.DataFrame({'src': df_res['src'],
	'dst': df_res['dst'],
	'weight': df_res['weight']})
g_e=spark.createDataFrame(g_e)
new_g=GraphFrame(g_v, g_e)

In [ ]:
#находим лидеров, вершины, степень которых равна максимальной или не меньше, чем 2/3 максимальной
degrees = new_g.degrees
unique_degree=degrees.select("degree").distinct()
from pyspark.sql.functions import max
df_max_degree=degrees.select(max(degrees.degree).alias("max_degree"))
df_max_degree.show()
row_max_degree=df_max_degree.select(["max_degree"]).collect()[0]
max_degree=row_max_degree[0]
leaders=degrees.filter((degrees.degree<=max_degree)&(degrees.degree>(max_degree*0.6)))
leaders=leaders.toPandas()

In [ ]:
vr = list(all_nodes)
ed=pd.DataFrame({'src': df_res['src'],
	'dst': df_res['dst'],
	'weight': df_res["weight"]})
edges=list(ed.itertuples(index=False, name=None))

I = ig.Graph.TupleList(edges, directed=True, weights=True)

l_weight_visualization = [f"{weight_for_formatting:.2f}" for weight_for_formatting in df_res["weight"]]

fig, ax = plt.subplots(figsize=(39, 45))
ig.plot(I, vertex_label=I.vs["name"], vertex_size=50, vertex_label_size=16, target=ax, edge_label=l_weight_visualization, edge_label_size=16)
fig.savefig("Граф только со связанными вершинами.png")
clusters=pd.DataFrame(columns=['id', 'cluster'])

In [ ]:
label=[]
label_follower=False
label_user=int(input("Введите идентификатор любого пользователя из таблицы 'ребра итогового графа с весами' в формате целого числа (просто убрать запятую): "))
#менять метку не будет пользователь c идентификатором, указанным в label_user и его подписчики, она у них будет фиксированной
fix_ver=[]
col_ver = I.vcount()
fix_index = I.vs.find(name=label_user).index
for v_index in range(col_ver):
	if v_index==fix_index:
		label.append(fix_index)
		fix_ver.append(True)
	else:
		v = int(I.vs[v_index]['name'])
		for row in range(df_res.shape[0]):
			if df_res['src'][row]==v and df_res['dst'][row]==label_user:
				label_follower=True
		if label_follower:
			label.append(fix_index)
			fix_ver.append(True)
		else:
			label.append(v_index)
			fix_ver.append(False)
	label_follower=False

In [ ]:
label_propogation_partition = I.community_label_propagation(weights="weight", initial=label, fixed=fix_ver)
result_label_propogation = label_propogation_partition.membership
lp_modularity=label_propogation_partition.modularity

clusters['id'] = I.vs['name']
clusters['cluster'] = result_label_propogation

unique_label_for_1=clusters['cluster'].unique()
print("Индексы получившихся кластеров")
print(unique_label_for_1)
print("Количество получившихся кластеров: ", len(unique_label_for_1))
print("Модулярность:", lp_modularity)

VersitesSortedByClusters = SortingByClusters(clusters)

print("Общие слова в постах в кластера:")
for clusters_ver in VersitesSortedByClusters:
	clusters_common_words = SearchForCommonWords(clusters_ver[1])
	print(clusters_ver[0], ":", clusters_common_words)
    
print("Часто встречающиеся слова в кластерах:")
for clusters_ver in VersitesSortedByClusters:
	clusters_common_words = SearchForFrequentWords(clusters_ver[1])
	print(clusters_ver[0], ":", clusters_common_words)
    
colors=ColorsGeneration(unique_label_for_1)
node_color=NodeColor(colors, clusters)
VisualizationOfIgraph(I, node_color, "Кластеризация алгоритмом распространения метки.png", "Кластеризация алгоритмом распространения метки")

In [ ]:
result_conected_components=new_g.stronglyConnectedComponents(maxIter=3)
result_conected_components.select("id","component").orderBy("component").show()

result_conected_components=result_conected_components.toPandas()
clusters['id']=result_conected_components['id']
clusters['cluster']=result_conected_components['component']

clusters=SortVer(clusters)
result_conected_components=clusters['cluster']
con_com_modularity = I.modularity(clusters['cluster'])

unique_label_for_2=clusters['cluster'].unique()
print("Индексы получишихся кластеров")
print(unique_label_for_2)
print("Количество получившихся кластеров:", len(unique_label_for_2))
print("Модулярность:", con_com_modularity)

VersitesSortedByClusters = SortingByClusters(clusters)

print("Общие слова в постах в кластера:")
for clusters_ver in VersitesSortedByClusters:
	clusters_common_words = SearchForCommonWords(clusters_ver[1])
	print(clusters_ver[0], ":", clusters_common_words)
    
print("Часто встречающиеся слова в кластерах:")
for clusters_ver in VersitesSortedByClusters:
	clusters_common_words = SearchForFrequentWords(clusters_ver[1])
	print(clusters_ver[0], ":", clusters_common_words)
    
colors=ColorsGeneration(unique_label_for_2)
node_color=NodeColor(colors, clusters)
VisualizationOfIgraph(I, node_color, "Кластеризация на основе выделения компонент связности.png", "Кластеризация на основе выделения компонент связности")

In [ ]:
from pyspark import StorageLevel
result_triangle=new_g.triangleCount(storage_level=StorageLevel(True, True, False, True, 1))
result_triangle=result_triangle.select("id", "count")
result_triangle.show()

In [ ]:
result_triangle=result_triangle.toPandas()
clusters['id']=result_triangle['id']
clusters['cluster']=result_triangle['count']

clusters=SortVer(clusters)
result_triangle = clusters['cluster']
t_modularity = I.modularity(clusters['cluster'])

unique_label_for_3=clusters['cluster'].unique()
print("Индексы получившихся кластеров")
print(unique_label_for_3)
print("Количество получившихся кластеров:", len(unique_label_for_3))
print("Модулярность:", t_modularity)

VersitesSortedByClusters = SortingByClusters(clusters)
print("Общие слова в постах в кластера:")
for clusters_ver in VersitesSortedByClusters:
	clusters_common_words = SearchForCommonWords(clusters_ver[1])
	print(clusters_ver[0], ":", clusters_common_words)
    
print("Часто встречающиеся слова в кластерах:")
for clusters_ver in VersitesSortedByClusters:
	clusters_common_words = SearchForFrequentWords(clusters_ver[1])
	print(clusters_ver[0], ":", clusters_common_words)
    
colors=ColorsGeneration(unique_label_for_3)
node_color=NodeColor(colors, clusters)
VisualizationOfIgraph(I, node_color, "Кластеризация подсчетом числа треугольников.png", "Кластеризация подсчетом числа треугольников")

In [ ]:
edge_betweenness_partition=I.community_edge_betweenness(directed=True,weights=I.es["weight"])
betweenness_communities=edge_betweenness_partition.as_clustering().membership
ed_modularity=edge_betweenness_partition.as_clustering().modularity

clusters['id'] = I.vs['name']
clusters['cluster'] = betweenness_communities

unique_label_for_4=clusters['cluster'].unique()
print("Индексы получившихся кластеров")
print(unique_label_for_4)
print("Количество получившихся кластеров:", len(unique_label_for_4))
print("Модулярность:", ed_modularity)

VersitesSortedByClusters = SortingByClusters(clusters)

print("Общие слова в постах в кластера:")
for clusters_ver in VersitesSortedByClusters:
	clusters_common_words = SearchForCommonWords(clusters_ver[1])
	print(clusters_ver[0], ":", clusters_common_words)
    
print("Часто встречающиеся слова в кластерах:")
for clusters_ver in VersitesSortedByClusters:
	clusters_common_words = SearchForFrequentWords(clusters_ver[1])
	print(clusters_ver[0], ":", clusters_common_words)
    
colors=ColorsGeneration(unique_label_for_4)
node_color=NodeColor(colors, clusters)
VisualizationOfIgraph(I, node_color, "Гирван-Ньюман igraph.png", "Кластеризация с помощью алгоритма Гирвана — Ньюмана igraph")

In [ ]:
import networkx as nx

G = nx.DiGraph() #создаем направленный граф для networkx
G.add_nodes_from(vr)
ed=pd.DataFrame({'src': df_res['src'],
	'dst': df_res['dst'],
	'weight': df_res["weight"]})
edges = [tuple(x) for x in ed.values]
G.add_weighted_edges_from(edges)

#алгоритм Гирвана-Ньюмана для networkx
gn_clusters = nx.community.girvan_newman(G) #начальные сообщества
best_mod = -1
gn_best_partition = None
for gn_community in gn_clusters:
	mod = nx.community.modularity(G, gn_community, weight='weight')
	if mod > best_mod:
		best_mod = mod
		gn_best_partition = gn_community
	else: break
gn_best_partition

In [ ]:
gn_modularity=best_mod

clusters=ClusterForVer(gn_best_partition)
clusters=SortVer(clusters)

girvan_newman_com=clusters['cluster']

unique_label_for_5=clusters['cluster'].unique()
print("Индексы получившихся кластеров")
print(unique_label_for_5)
print("Количество получившихся кластеров:", len(unique_label_for_5))
print("Модулярность:", gn_modularity)

VersitesSortedByClusters = SortingByClusters(clusters)

print("Общие слова в постах в кластера:")
for clusters_ver in VersitesSortedByClusters:
	clusters_common_words = SearchForCommonWords(clusters_ver[1])
	print(clusters_ver[0], ":", clusters_common_words)
    
print("Часто встречающиеся слова в кластерах:")
for clusters_ver in VersitesSortedByClusters:
	clusters_common_words = SearchForFrequentWords(clusters_ver[1])
	print(clusters_ver[0], ":", clusters_common_words)
    
colors=ColorsGeneration(unique_label_for_5)
node_color=NodeColor(colors, clusters)
VisualizationOfIgraph(I, node_color, "Гирван-Ньюман networkx.png", "Кластеризация с помощью алгоритма Гирвана — Ньюмана networkx")

In [ ]:
import leidenalg
leiden_partition = leidenalg.find_partition(I, leidenalg.ModularityVertexPartition, weights='weight')
leiden_communities = leiden_partition.membership
le_la_mod=leiden_partition.modularity

clusters['id']=I.vs['name']
clusters['cluster']=leiden_communities

unique_label_for_6=clusters['cluster'].unique()
print("Индексы получившихся кластеров")
print(unique_label_for_6)
print("Количество получившихся кластеров:", len(unique_label_for_6))
print("Модулярность:", le_la_mod)

VersitesSortedByClusters = SortingByClusters(clusters)

print("Общие слова в постах в кластера:")
for clusters_ver in VersitesSortedByClusters:
	clusters_common_words = SearchForCommonWords(clusters_ver[1])
	print(clusters_ver[0], ":", clusters_common_words)
    
print("Часто встречающиеся слова в кластерах:")
for clusters_ver in VersitesSortedByClusters:
	clusters_common_words = SearchForFrequentWords(clusters_ver[1])
	print(clusters_ver[0], ":", clusters_common_words)
    
colors=ColorsGeneration(unique_label_for_6)
node_color=NodeColor(colors, clusters)
VisualizationOfIgraph(I, node_color, "Лейден leidenalg.png", "Кластеризация алгоритмом Лейдена leidenalg")

In [ ]:
optimiser = leidenalg.Optimiser()
#находим разбиение вершин на сообщества
louvain_partition = leidenalg.ModularityVertexPartition(I, weights='weight')
#агрегируем вершины в узлы
agg_partition = louvain_partition.aggregate_partition()
while optimiser.move_nodes(agg_partition) > 0:
	#обновляем разбиение узлов на сообщества
	louvain_partition.from_coarse_partition(agg_partition)
	agg_partition = agg_partition.aggregate_partition()
louvain_la_comm=louvain_partition.membership
lv_la_mod=louvain_partition.modularity

clusters['id']=I.vs['name']
clusters['cluster']=louvain_la_comm

unique_label_for_7=clusters['cluster'].unique()
print("Индексы получившихся кластеров")
print(unique_label_for_7)
print("Количество получившихся кластеров:", len(unique_label_for_7))
print("Модулярность:", lv_la_mod)

VersitesSortedByClusters = SortingByClusters(clusters)

print("Общие слова в постах в кластера:")
for clusters_ver in VersitesSortedByClusters:
	clusters_common_words = SearchForCommonWords(clusters_ver[1])
	print(clusters_ver[0], ":", clusters_common_words)
    
print("Часто встречающиеся слова в кластерах:")
for clusters_ver in VersitesSortedByClusters:
	clusters_common_words = SearchForFrequentWords(clusters_ver[1])
	print(clusters_ver[0], ":", clusters_common_words)
    
colors=ColorsGeneration(unique_label_for_7)
node_color=NodeColor(colors, clusters)
VisualizationOfIgraph(I, node_color, "Лувен leidenalg.png", "Кластеризация алгоритмом Лувена leidenalg")

In [ ]:
partition=nx.community.louvain_communities(G, weight='weight')
lv_modularity = nx.community.modularity(G, partition, weight='weight')

clusters=ClusterForVer(partition)
clusters=SortVer(clusters)

louvain_classes=clusters['cluster']

unique_label_for_8=clusters['cluster'].unique()
print("Индексы получившихся кластеров")
print(unique_label_for_8)
print("Количество получившихся кластеров:", len(unique_label_for_8))
print("Модулярность:", lv_modularity)

VersitesSortedByClusters = SortingByClusters(clusters)

print("Общие слова в постах в кластера:")
for clusters_ver in VersitesSortedByClusters:
	clusters_common_words = SearchForCommonWords(clusters_ver[1])
	print(clusters_ver[0], ":", clusters_common_words)
    
print("Часто встречающиеся слова в кластерах:")
for clusters_ver in VersitesSortedByClusters:
	clusters_common_words = SearchForFrequentWords(clusters_ver[1])
	print(clusters_ver[0], ":", clusters_common_words)
    
colors=ColorsGeneration(unique_label_for_8)
node_color=NodeColor(colors, clusters)
VisualizationOfIgraph(I, node_color, "Лувен networkx.png", "Кластеризация алгоритмом Лувена networkx")

In [ ]:
user_posts=[]
for user_id in I.vs['name']:
	for row in range(len(df_users)):
		if (int(user_id)==df_users['id'][row]):
			user_posts.append(df_users['post'][row])
			break

In [ ]:
comparison_of_clustering_methods=pd.DataFrame({
	'user_id': vr,
	'posts': user_posts,
	'louvain':louvain_classes,
	'louvain (leidenalg)': louvain_la_comm,
	'leiden (leidenalg)': leiden_communities,
	'community edge betweenness (girvan newman igraph)':betweenness_communities,
	'girvan newman':girvan_newman_com,
	'label propogation': result_label_propogation,
	'conected components': result_conected_components,
	'triangle count': result_triangle})
comparison_of_clustering_methods.to_excel('результаты применения различных методов кластеризации.xlsx', index=False)